# TwinStock AI — IBM Granite TTM R2.1 Experimentation Notebook

This notebook demonstrates zero-shot demand forecasting using the IBM Granite Time Series model (TTM R2.1).

**It is standalone — the ML service API does NOT depend on this notebook.**

## Covered:
1. Load synthetic warehouse demand data
2. Validate and preprocess the time series
3. Load IBM Granite TTM R2.1 (`ibm-granite/granite-timeseries-ttm-r2`, revision `90-30-ft-r2.1`)
4. Run zero-shot daily demand forecasting
5. Plot historical demand vs Granite forecast
6. Compare Baseline Moving Average vs Granite (MAE, RMSE, MAPE)
7. Demonstrate the API request format

---

### Requirements
```
pip install torch transformers datasets accelerate scipy scikit-learn
```
Plus `tsfm_public` — see README for installation instructions.

### Important notes
- `data/warehouse_raw.csv` is an **inventory snapshot**, NOT historical demand. It is NOT used here.
- Synthetic demand is generated for demonstration only.
- Do NOT claim this synthetic data represents real warehouse performance.

In [ ]:
import sys
from pathlib import Path

# Ensure ml-service root is on sys.path
root = Path('.').resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import datetime
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('Python:', sys.version)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('Imports OK')

## Step 1 — Generate Synthetic Warehouse Demand

We need at least **90 days** of daily demand for the Granite TTM 90-30-ft-r2.1 model.

In [ ]:
N_DAYS = 120  # more than the 90-day context window
ITEM_ID = 'ITM10025'

rng = np.random.default_rng(seed=42)
start = datetime.date(2026, 1, 1)
dates = [start + datetime.timedelta(days=i) for i in range(N_DAYS)]

# Realistic warehouse demand: weekly seasonality + trend + noise
t = np.arange(N_DAYS)
demand = (
    20.0                                      # baseline
    + 3.0 * np.sin(2 * np.pi * t / 7)        # weekly cycle
    + 0.05 * t                                # slight upward trend
    + rng.normal(0, 2.0, N_DAYS)              # noise
)
demand = np.clip(demand, 0, None).round(1)

history = [
    {'date': str(d), 'demand': float(v)}
    for d, v in zip(dates, demand)
]

df = pd.DataFrame(history)
print(f'Dataset: {len(df)} rows  |  date range: {df["date"].min()} → {df["date"].max()}')
print(f'Demand stats: mean={df["demand"].mean():.1f}  std={df["demand"].std():.1f}  min={df["demand"].min():.1f}  max={df["demand"].max():.1f}')
df.head()

## Step 2 — Preprocess the Time Series

In [ ]:
from src.preprocessing import prepare_time_series
from src.evaluation import chronological_train_test_split, evaluate_forecast

# Add item_id column for preprocessing utilities
df['item_id'] = ITEM_ID
processed = prepare_time_series(df)

print(f'After preprocessing: {len(processed)} rows')
print(f'Columns: {list(processed.columns)}')

# Convert back to list format used by the forecast API
all_history = [
    {'date': str(row['date'].date() if hasattr(row['date'], 'date') else row['date']),
     'demand': float(row['demand'])}
    for _, row in processed.iterrows()
]

# Chronological 70/30 split (NO random shuffling)
train_history, test_history = chronological_train_test_split(all_history, test_ratio=0.25)
print(f'Train: {len(train_history)} days  |  Test: {len(test_history)} days')

## Step 3 — Load IBM Granite TTM R2.1

Model: `ibm-granite/granite-timeseries-ttm-r2`  
Revision: `90-30-ft-r2.1`  

- context_length = 90 days  
- prediction_length = 30 days  
- No fine-tuning needed — zero-shot inference  
- Weights: ~3 MB (cached after first download)

In [ ]:
from src.forecast import GraniteForecastModel, BaselineMovingAverageModel

granite = GraniteForecastModel()
loaded = granite.load()

if loaded:
    print(f'✓ Granite TTM loaded successfully')
    print(f'  Model: {granite.model_name}')
    print(f'  Available: {granite.is_available}')
    print(f'  Min history required: {granite.MIN_HISTORY} days')
else:
    print(f'✗ Granite failed to load: {granite.load_error}')
    print('Continuing with baseline model only.')

## Step 4 — Generate Forecast

We forecast the next **7 days** (or as many as the test set contains).

In [ ]:
HORIZON = min(7, len(test_history))

# Baseline forecast
baseline = BaselineMovingAverageModel(window=7)
baseline_forecast = baseline.predict(train_history, horizon=HORIZON)
print(f'Baseline ({baseline.model_name}) — {HORIZON} day forecast:')
for pt in baseline_forecast:
    print(f'  {pt["date"]}: {pt["predicted_demand"]:.2f}')

print()

# Granite forecast (uses the last 90 days of train_history as context)
granite_forecast = None
if granite.is_available and len(train_history) >= granite.MIN_HISTORY:
    granite_forecast = granite.predict(train_history, horizon=HORIZON)
    print(f'Granite TTM ({granite.model_name}) — {HORIZON} day forecast:')
    for pt in granite_forecast:
        print(f'  {pt["date"]}: {pt["predicted_demand"]:.4f}')
elif len(train_history) < granite.MIN_HISTORY if granite.is_available else False:
    print(f'⚠ Granite needs {granite.MIN_HISTORY} days; only {len(train_history)} provided.')
else:
    print('Granite model not available — baseline only.')

## Step 5 — Plot Historical Demand vs Forecast

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

# Historical (training) demand — show last 30 days for clarity
show_n = 30
train_dates = [pd.to_datetime(p['date']) for p in train_history[-show_n:]]
train_vals  = [p['demand'] for p in train_history[-show_n:]]
ax.plot(train_dates, train_vals, label='Historical demand (train)', color='steelblue', linewidth=1.5)

# Actual test demand
test_dates = [pd.to_datetime(p['date']) for p in test_history[:HORIZON]]
test_vals  = [p['demand'] for p in test_history[:HORIZON]]
ax.plot(test_dates, test_vals, label='Actual demand (test)', color='green', linewidth=1.5, linestyle='--')

# Baseline forecast
baseline_dates = [pd.to_datetime(p['date']) for p in baseline_forecast]
baseline_vals  = [p['predicted_demand'] for p in baseline_forecast]
ax.plot(baseline_dates, baseline_vals, label=f'Forecast: {baseline.model_name}', color='orange', linewidth=2, marker='o', markersize=4)

# Granite forecast (if available)
if granite_forecast:
    granite_dates = [pd.to_datetime(p['date']) for p in granite_forecast]
    granite_vals  = [p['predicted_demand'] for p in granite_forecast]
    ax.plot(granite_dates, granite_vals, label=f'Forecast: {granite.model_name}', color='red', linewidth=2, marker='s', markersize=4)

ax.set_title(f'Warehouse Demand Forecast — Item: {ITEM_ID}', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Demand (units)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 6 — Evaluate Forecasts (MAE, RMSE, MAPE)

**Chronological split — no random shuffling.**

In [ ]:
actual_test = test_history[:HORIZON]

print('=== Forecast Evaluation ===')
print(f'Horizon: {HORIZON} days  |  Test points: {len(actual_test)}')
print()

# Baseline evaluation
base_metrics = evaluate_forecast(actual_test, baseline_forecast)
print(f'--- {baseline.model_name} ---')
print(f'  MAE:  {base_metrics["mae"]}')
print(f'  RMSE: {base_metrics["rmse"]}')
print(f'  MAPE: {base_metrics["mape"]}%')
print()

# Granite evaluation
if granite_forecast:
    granite_metrics = evaluate_forecast(actual_test, granite_forecast)
    print(f'--- {granite.model_name} ---')
    print(f'  MAE:  {granite_metrics["mae"]}')
    print(f'  RMSE: {granite_metrics["rmse"]}')
    print(f'  MAPE: {granite_metrics["mape"]}%')
    print()
    print('NOTE: Evaluation on synthetic data only — does not represent real warehouse accuracy.')
else:
    print('Granite not available — cannot compare.')
    print('NOTE: Evaluation requires actual historical demand from the backend.')

## Step 7 — API Request Format

This is how the backend calls `POST /forecast`:

In [ ]:
from src.forecast import forecast_demand
import json

# Show what the request/response looks like
sample_request = {
    'item_id': ITEM_ID,
    'horizon': HORIZON,
    'history': train_history  # 90+ days from backend
}

result = forecast_demand(
    item_id=sample_request['item_id'],
    historical_data=sample_request['history'],
    horizon=sample_request['horizon'],
)

print('Response from forecast_demand():')
print(json.dumps({
    'item_id': result['item_id'],
    'model': result['model'],
    'horizon': result['horizon'],
    'forecast': result['forecast'][:3],  # first 3 days
    '...': f'({HORIZON - 3} more days)'
}, indent=2))

print()
print('Active model:', result['model'])
print('Predictions are', 'real Granite TTM output' if 'Granite' in result['model'] else 'baseline moving average (Granite not loaded)')

## Model Information

In [ ]:
from src.forecast import get_model_status
import json

status = get_model_status()
print('GET /model/status response:')
print(json.dumps(status, indent=2))

---

## Summary

| Property | Value |
|---|---|
| Model | IBM Granite TTM R2.1 |
| HuggingFace repo | `ibm-granite/granite-timeseries-ttm-r2` |
| Revision | `90-30-ft-r2.1` |
| Context length | 90 days |
| Max forecast horizon | 30 days |
| Frequency | Daily |
| Fine-tuning needed | No — zero-shot |
| External scaling | No — internal normalization |

**This notebook is for experimentation only.**  
The production forecasting API is at `http://127.0.0.1:8001/docs`.